# Session V: File I/O and Structured Data

Persist data using text files, JSON, and CSV, and learn how to design file-based mini databases.

## Exercise 1: Daily Journal App
**Objective:** Practice reading, writing, and appending to text files.

Build a simple daily journal application that stores entries in a plain text file (`journal.txt`). Each entry should include a timestamp and the user's text.

**Tasks:**
1. `add_entry(filename, text)` → Append a new journal entry with the current date/time to the file.
2. `read_journal(filename)` → Read and display all entries from the file. Handle the case where the file doesn't exist yet.
3. `search_journal(filename, keyword)` → Read the file and return only lines that contain the keyword (case-insensitive).
4. `count_entries(filename)` → Return the total number of entries in the journal.
5. `run_journal()` → A menu-driven interface: [A]dd entry, [R]ead all, [S]earch, [C]ount, [Q]uit.

In [3]:
from datetime import datetime

JOURNAL_FILE = "journal.txt"

def add_entry(filename, text):
    """Append a timestamped journal entry to the file."""
    # TODO: Open file in append mode
    # TODO: Write the current datetime + the entry text
    with open(filename, 'a') as file:
        now = datetime.now()
        entry = f"[{now:%Y-%m-%d %H:%M:%S}] {text}\n"
        file.write(entry)

def read_journal(filename):
    """Read and print all entries. Handle FileNotFoundError."""
    # TODO: Try to open and read the file
    # TODO: Print each line (stripped)
    # TODO: Handle missing file gracefully
    try:
        with open(filename, 'r') as file:
            for line in file:
                print(line.strip())
    except FileNotFoundError:
        print("File not found")

def search_journal(filename, keyword):
    """Return a list of lines containing the keyword (case-insensitive)."""
    # TODO: Read all lines
    # TODO: Filter lines that contain the keyword
    # TODO: Handle missing file
    lower_keyword = keyword.lower()
    try:
        with open(filename, 'r') as file:
            return [line.strip() for line in file if lower_keyword in line.lower()]
    except FileNotFoundError:
        print("File not found")
        return []

def count_entries(filename):
    """Count and return the number of journal entries."""
    # TODO: Read lines and count non-empty entries
    # TODO: Handle missing file (return 0)
    try:
        with open(filename, 'r') as file:
            return sum(1 for line in file if line.strip())
    except FileNotFoundError:
        print("File not found")
        return 0

def run_journal():
    """Menu-driven journal interface."""
    # TODO: while True loop with menu [A]dd [R]ead [S]earch [C]ount [Q]uit
    # TODO: Route to the correct function based on user input
    allowed = {'A', 'R', 'S', 'C', 'Q'}
    while True:
        print("\n ******** Daily Journal ********")
        print("[A]dd entry")
        print("[R]ead Journal")
        print("[S]earch Journal")
        print("[C]ount entries")
        print("[Q]uit Journal")
        user_input = input("Choose:").strip().upper()
        if user_input == 'Q':
            print("Thank you for using!")
            break
        elif user_input not in allowed:
            print("Please Enter the correct input")
            continue
        if user_input == 'A':
            line = input("Enter the Text you want to add:")
            add_entry(JOURNAL_FILE, line)
        elif user_input == 'R':
            read_journal(JOURNAL_FILE)
        elif user_input == 'S':
            keyword = input("Enter the keyword you want to search:").strip()
            results = search_journal(JOURNAL_FILE, keyword)
            if results:
                print(f"\nFound {len(results)} matching entires for {keyword}:")
                for res in results:
                    print("\t", res)
            else:
                print(f"\nNo matching entires found for {keyword}")
        elif user_input == 'C':
            print("Total entries:", count_entries(JOURNAL_FILE))

# --- QUICK TEST (non-interactive) ---
add_entry(JOURNAL_FILE, "Started learning about file I/O today!")
add_entry(JOURNAL_FILE, "Python makes reading files so easy.")
read_journal(JOURNAL_FILE)
print("Search results:", search_journal(JOURNAL_FILE, "python"))
print("Total entries:", count_entries(JOURNAL_FILE))

run_journal()  # Uncomment to test the interactive menu

[2026-10-18 08:21:36] Started learning about file I/O today!
[2026-10-18 08:21:36] Python makes reading files so easy.
Search results: ['[2026-10-18 08:21:36] Python makes reading files so easy.']
Total entries: 2

 ******** Daily Journal ********
[A]dd entry
[R]ead Journal
[S]earch Journal
[C]ount entries
[Q]uit Journal



 ******** Daily Journal ********
[A]dd entry
[R]ead Journal
[S]earch Journal
[C]ount entries
[Q]uit Journal
[2026-10-18 08:21:36] Started learning about file I/O today!
[2026-10-18 08:21:36] Python makes reading files so easy.
[2026-10-18 08:21:49] Hello World I am Python

 ******** Daily Journal ********
[A]dd entry
[R]ead Journal
[S]earch Journal
[C]ount entries
[Q]uit Journal

Found 1 matching entires for Hello:
	 [2026-10-18 08:21:49] Hello World I am Python

 ******** Daily Journal ********
[A]dd entry
[R]ead Journal
[S]earch Journal
[C]ount entries
[Q]uit Journal
Total entries: 3

 ******** Daily Journal ********
[A]dd entry
[R]ead Journal
[S]earch Journal
[C]ount entries
[Q]uit Journal
Thank you for using!


---
## Exercise 2: To-Do List Manager (JSON Persistence)
**Objective:** Build a CRUD application that stores data in a JSON file.

Create a to-do list manager where tasks are stored as a list of dictionaries in a `tasks.json` file. Each task has a `title`, a `priority` (high/medium/low), and a `done` status (True/False).

**Tasks:**
1. `load_tasks(filename)` → Load tasks from JSON. Return an empty list if the file is missing or corrupted.
2. `save_tasks(filename, tasks)` → Save the tasks list to JSON with pretty formatting.
3. `add_task(tasks)` → Prompt for title and priority, append a new task dict (done=False).
4. `list_tasks(tasks)` → Display all tasks in a formatted table with their index, status, title, and priority.
5. `mark_done(tasks)` → Ask for a task index and set its `done` field to True.
6. `delete_task(tasks)` → Ask for a task index and remove the task from the list.
7. `run_todo()` → Menu loop: [A]dd, [L]ist, [D]one, [R]emove, [Q]uit. Auto-save after every change.

**Data structure:**
```python
[
    {"title": "Buy groceries", "priority": "high", "done": False},
    {"title": "Read chapter 5", "priority": "medium", "done": True}
]
```

In [3]:
import json

TODO_FILE = "tasks.json"

def load_tasks(filename):
    """Load tasks from a JSON file. Return [] if missing or corrupted."""
    # TODO: try/except for FileNotFoundError and json.JSONDecodeError
    try:
        with open(filename, "r") as file:
            tasks = json.load(file)
        return tasks
    except (FileNotFoundError, json.JSONDecodeError):
        return []

def save_tasks(filename, tasks):
    """Save tasks list to a JSON file with indent=2."""
    # TODO: Write tasks to file using json.dump
    with open(filename, "w") as file:
        json.dump(tasks, file, indent=2)

def add_task(tasks):
    """Prompt for title and priority, append new task."""
    # TODO: Get title from user (reject empty)
    # TODO: Get priority (validate: high/medium/low)
    # TODO: Append {"title": ..., "priority": ..., "done": False}
    title = ""
    while not title:
        title = input("Enter the Title: ").strip()
    priority = ""
    allowed = {"high", "medium", "low"}
    while priority not in allowed:
        priority = input("Enter priority (validate: high/medium/low)").strip().lower()

    tasks.append({"title": title, "priority": priority, "done": False})
    return tasks

def list_tasks(tasks):
    """Display all tasks in a formatted table."""
    # TODO: Handle empty list
    # TODO: Loop with enumerate, show index, done status, title, priority
    if not tasks:
        print("There are no Tasks in the file")
        return
    for index, task in enumerate(tasks, start=1):
        status = "Done" if task['done'] else "Not Done"
        print(f"{index:<4} {status:<10} {task['priority']:<10} {task['title']}")

def mark_done(tasks):
    """Ask for a task index and mark it as done."""
    # TODO: Validate the index (try/except + range check)
    # TODO: Set tasks[index]["done"] = True
    if not tasks:
        print("There are no tasks in the file")
        return
    
    while True:
        index = input(f"Enter Task Index: 1 - {len(tasks)}").strip()
        try:
            index = int(index)
            if 1 <= index <= len(tasks):
                tasks[index - 1]['done'] = True
                break
            else:
                print(f"Enter a valid length (1 - {len(tasks)})")
        except ValueError:
            print("Invalid input format! Please enter a number")

def delete_task(tasks):
    """Ask for a task index and remove the task."""
    # TODO: Validate the index
    # TODO: Remove with .pop()
    if not tasks:
        print("There are no tasks in the file")
        return
        
    while True:
        index = input(f"Enter Task Index: 1 - {len(tasks)}").strip()
        try:
            index = int(index)
            if 1 <= index <= len(tasks):
                removed = tasks.pop(index - 1)
                print(f"Removed Task #{index} {removed['title']}")
                break
            else:
                print(f"Enter a valid length (1 - {len(tasks)})")
        except ValueError:
            print("Invalid input format! Please enter a number")

def run_todo():
    """Main menu loop with auto-save."""
    # TODO: Load tasks at start
    # TODO: while True menu: [A]dd [L]ist [D]one [R]emove [Q]uit
    # TODO: Save after every modification
    tasks = load_tasks(TODO_FILE)
    allowed = {'A', 'L', 'D', 'R', 'Q'}
    while True:
        print("\n ******** Tasks ********")
        print("[A]dd")
        print("[L]ist")
        print("[D]one")
        print("[R]emove")
        print("[Q]uit")
        user_input = input("Choose:").strip().upper()
        if user_input == 'Q':
            print("Thank you for using!")
            break
        elif user_input not in allowed:
            print("Please Enter the correct input")
            continue
        if user_input == 'A':
            add_task(tasks)
            save_tasks(TODO_FILE, tasks)
        elif user_input == 'L':
            list_tasks(tasks)
        elif user_input == 'D':
            mark_done(tasks)
            save_tasks(TODO_FILE, tasks)
        elif user_input == 'R':
            delete_task(tasks)
            save_tasks(TODO_FILE, tasks)


# # --- QUICK TEST (non-interactive) ---
# tasks = load_tasks(TODO_FILE)
# # tasks.append({"title": "Test task", "priority": "high", "done": False})
# # save_tasks(TODO_FILE, tasks)
# # tasks = load_tasks(TODO_FILE)
# # add_task(tasks)
# # save_tasks(TODO_FILE, tasks)
# # mark_done(tasks)
# delete_task(tasks)
# save_tasks(TODO_FILE, tasks)
# list_tasks(tasks)

run_todo()  # Uncomment to test the interactive menu


 ******** Tasks ********
[A]dd
[L]ist
[D]one
[R]emove
[Q]uit

 ******** Tasks ********
[A]dd
[L]ist
[D]one
[R]emove
[Q]uit
1    Not Done   low        Python

 ******** Tasks ********
[A]dd
[L]ist
[D]one
[R]emove
[Q]uit

 ******** Tasks ********
[A]dd
[L]ist
[D]one
[R]emove
[Q]uit
1    Done       low        Python

 ******** Tasks ********
[A]dd
[L]ist
[D]one
[R]emove
[Q]uit

 ******** Tasks ********
[A]dd
[L]ist
[D]one
[R]emove
[Q]uit

 ******** Tasks ********
[A]dd
[L]ist
[D]one
[R]emove
[Q]uit

 ******** Tasks ********
[A]dd
[L]ist
[D]one
[R]emove
[Q]uit
1    Done       low        Python
2    Not Done   high       Mobile App Development
3    Not Done   high       Python Assignment
4    Not Done   medium     Sql assignment

 ******** Tasks ********
[A]dd
[L]ist
[D]one
[R]emove
[Q]uit
Thank you for using!


---
## Exercise 3: CSV Expense Importer
**Objective:** Read, validate, and analyze data from a CSV file.

You receive a CSV file of monthly expenses. Some rows may have missing or invalid data. Write functions to import the data safely, clean it, and produce summary reports.

**First, run the cell below to generate the sample CSV file.** Then write the functions to process it.

**Tasks:**
1. `load_expenses(filename)` → Read the CSV using `csv.DictReader`. Validate each row: skip rows where `amount` is not a valid number or `category` is empty. Return a list of clean dicts (with `amount` converted to float).
2. `total_by_category(expenses)` → Return a dict mapping each category to its total spending.
3. `most_expensive(expenses)` → Return the single most expensive entry.
4. `print_report(expenses)` → Print a formatted summary: total spent, breakdown by category, and the most expensive item.

In [5]:
import csv

# --- RUN THIS CELL FIRST to generate the sample CSV ---
sample_data = """date,description,category,amount
2024-01-05,Grocery store,Food,52.30
2024-01-06,Bus pass,Transport,40.00
2024-01-07,Coffee shop,Food,4.50
2024-01-08,Netflix,Entertainment,15.99
2024-01-09,Gas station,Transport,bad_data
2024-01-10,Restaurant,Food,38.75
2024-01-11,Gym membership,,45.00
2024-01-12,Bookstore,Education,22.50
2024-01-13,Electricity bill,Utilities,89.00
2024-01-14,Movie tickets,Entertainment,24.00
2024-01-15,Taxi,Transport,18.50
2024-01-16,Online course,Education,49.99
2024-01-17,Supermarket,Food,67.20
2024-01-18,Water bill,Utilities,not_a_number
2024-01-19,Concert,Entertainment,75.00
2024-01-20,Pharmacy,Health,12.40"""

with open("expenses.csv", "w", newline="") as f:
    f.write(sample_data)

print("✅ expenses.csv created successfully.")

✅ expenses.csv created successfully.


In [11]:
import csv
from collections import defaultdict

def load_expenses(filename):
    """Load and validate expenses from CSV. Skip invalid rows."""
    # TODO: Open file with csv.DictReader
    # TODO: For each row, validate that 'amount' converts to float
    # TODO: Validate that 'category' is not empty
    # TODO: Return list of clean dicts with amount as float
    clean_expenses = []
    with open(filename, 'r') as file:
        reader = csv.DictReader(file)
        for row in reader:
            category = row.get("category", "").strip()
            if not category:
                continue
            try:
                amount = float(row["amount"])
                clean_expenses.append({
                                **row,
                                "category": category,
                                "amount": amount
                            })
            except (ValueError, TypeError):
                continue
    return clean_expenses

def total_by_category(expenses):
    """Return {category: total_amount} dict."""
    # TODO: Accumulation pattern over categories
    by_category = defaultdict(float)
    for expense in expenses:
        by_category[expense['category']] += expense['amount']
    return dict(by_category)

def most_expensive(expenses):
    """Return the expense dict with the highest amount."""
    # TODO: Use max() with a key, or loop
    return max(expenses, key=lambda x: x['amount'], default=None)

def print_report(expenses):
    """Print a formatted expense summary report."""
    # TODO: Total spent
    # TODO: Breakdown by category
    # TODO: Most expensive item
    if not expenses:
        print("No Expenses to Report!")
        return
    total = sum(e['amount'] for e in expenses)
    print(f"Total Spending: {total}")
    by_category = total_by_category(expenses)
    print("Spending by Category")
    for cat, val in by_category.items():
        print(cat, val)
    print(f"Most Expensive Expense: {most_expensive(expenses)}")


# --- TEST ---
expenses = load_expenses("expenses.csv")
print_report(expenses)

Total Spending: 510.13
Spending by Category
Food 162.75
Transport 58.5
Entertainment 114.99000000000001
Education 72.49000000000001
Utilities 89.0
Health 12.4
Most Expensive Expense: {'date': '2024-01-13', 'description': 'Electricity bill', 'category': 'Utilities', 'amount': 89.0}


---
## Exercise 4: File Content Validator
**Objective:** Write robust validation functions for both JSON and CSV files.

You will build a validation toolkit that checks whether files are well-formed and contain the expected structure *before* your application tries to use the data.

**First, run the cell below to generate sample test files** (some valid, some intentionally broken). Then write the validators.

**Tasks:**
1. `validate_json(filename, required_keys)` → Check that the file (a) exists, (b) contains valid JSON, (c) is a list of dicts, and (d) each dict contains all `required_keys`. Return a report dict: `{"valid": int, "invalid": int, "errors": [str]}`.
2. `validate_csv(filename, expected_columns)` → Check that the file (a) exists, (b) has the correct number of columns in the header, and (c) each data row has the correct number of fields. Return a similar report dict.
3. `print_validation_report(report, filename)` → Pretty-print the validation results.

In [12]:
import json

# --- RUN THIS CELL FIRST to generate test files ---

# Good JSON file
good_json = [
    {"name": "Alice", "age": 22, "email": "alice@test.com"},
    {"name": "Bob", "age": 25, "email": "bob@test.com"},
    {"name": "Charlie", "age": 30, "email": "charlie@test.com"},
]
with open("good_data.json", "w") as f:
    json.dump(good_json, f, indent=2)

# Bad JSON file (missing keys, wrong types)
bad_json = [
    {"name": "Alice", "age": 22, "email": "alice@test.com"},
    {"name": "Bob", "age": 25},
    "this is not a dict",
    {"name": "Diana", "age": "not_a_number", "email": "diana@test.com"},
]
with open("bad_data.json", "w") as f:
    json.dump(bad_json, f, indent=2)

# Corrupted JSON file
with open("corrupted.json", "w") as f:
    f.write('{"name": "broken", missing_quotes: true}')

# Good CSV file
with open("good_data.csv", "w", newline="") as f:
    f.write("name,age,email\n")
    f.write("Alice,22,alice@test.com\n")
    f.write("Bob,25,bob@test.com\n")

# Bad CSV file (inconsistent columns)
with open("bad_data.csv", "w", newline="") as f:
    f.write("name,age,email\n")
    f.write("Alice,22,alice@test.com\n")
    f.write("Bob,25\n")
    f.write("Charlie,30,charlie@test.com,extra_field\n")
    f.write("Diana,28,diana@test.com\n")

print("✅ All test files created successfully.")

✅ All test files created successfully.


In [15]:
import json
import csv

def validate_json(filename, required_keys):
    """Validate a JSON file's structure and required keys.
    Return {"valid": int, "invalid": int, "errors": [str]}."""
    # TODO: Try to open and parse the file
    # TODO: Check that root is a list
    # TODO: Check each item is a dict with all required_keys
    # TODO: Collect errors and counts
    report = {"valid": 0, "invalid": 0, "errors": []}

    try:
        with open(filename, "r", encoding="utf-8") as file:
            data = json.load(file)
    except FileNotFoundError:
        report["errors"].append(f"File not found: '{filename}'")
        return report
    except json.JSONDecodeError as e:
        report["errors"].append(f"Invalid JSON syntax: {e.msg} (line {e.lineno}, col {e.colno})")
        return report

    if not isinstance(data, list):
        report["errors"].append(f"Root structure must be a list, got {type(data).__name__}")
        return report

    for idx, item in enumerate(data, start=1):
        if not isinstance(item, dict):
            report["invalid"] += 1
            report["errors"].append(f"Record #{idx}: Expected dict, got {type(item).__name__}")
            continue

        missing = [k for k in required_keys if k not in item]
        if missing:
            report["invalid"] += 1
            report["errors"].append(f"Record #{idx}: Missing required keys {missing}")
        else:
            report["valid"] += 1

    return report

def validate_csv(filename, expected_columns):
    """Validate a CSV file's structure and column count.
    Return {"valid": int, "invalid": int, "errors": [str]}."""
    # TODO: Try to open the file
    # TODO: Check header column count
    # TODO: Check each row's column count
    # TODO: Collect errors and counts
    expected_count = (
        expected_columns if isinstance(expected_columns, int) else len(expected_columns)
    )

    try:
        with open(filename, "r", encoding="utf-8", newline="") as file:
            reader = csv.reader(file)
            header = next(reader, None)

            if header is None:
                report["errors"].append("File is empty")
                return report

            if len(header) != expected_count:
                report["errors"].append(
                    f"Header column count mismatch: expected {expected_count}, got {len(header)}"
                )

            for row_num, row in enumerate(reader, start=2):
                if not row:
                    continue
                if len(row) == expected_count:
                    report["valid"] += 1
                else:
                    report["invalid"] += 1
                    report["errors"].append(
                        f"Row {row_num}: Expected {expected_count} columns, found {len(row)}"
                    )

    except FileNotFoundError:
        report["errors"].append(f"File not found: '{filename}'")
    except Exception as e:
        report["errors"].append(f"Read error: {e}")

    return report

def print_validation_report(report, filename):
    """Pretty-print the validation results."""
    # TODO: Print filename, valid/invalid counts, and error details
    status = "PASSED" if not report["errors"] and report["invalid"] == 0 else "FAILED"
    print(f"File: {filename}")
    print(f"Status: {status}")
    print(f"  Valid Records:   {report['valid']}")
    print(f"  Invalid Records: {report['invalid']}")

    if report["errors"]:
        print(f"  Errors Found ({len(report['errors'])}):")
        for err in report["errors"]:
            print(f"    • {err}")
    else:
        print("  Errors: None")


# --- TEST ---
print("=" * 50)
report = validate_json("good_data.json", ["name", "age", "email"])
print_validation_report(report, "good_data.json")

print("=" * 50)
report = validate_json("bad_data.json", ["name", "age", "email"])
print_validation_report(report, "bad_data.json")

print("=" * 50)
report = validate_json("corrupted.json", ["name", "age", "email"])
print_validation_report(report, "corrupted.json")

print("=" * 50)
report = validate_csv("good_data.csv", 3)
print_validation_report(report, "good_data.csv")

print("=" * 50)
report = validate_csv("bad_data.csv", 3)
print_validation_report(report, "bad_data.csv")

File: good_data.json
Status: PASSED
  Valid Records:   3
  Invalid Records: 0
  Errors: None
File: bad_data.json
Status: FAILED
  Valid Records:   2
  Invalid Records: 2
  Errors Found (2):
    • Record #2: Missing required keys ['email']
    • Record #3: Expected dict, got str
File: corrupted.json
Status: FAILED
  Valid Records:   0
  Invalid Records: 0
  Errors Found (1):
    • Invalid JSON syntax: Expecting property name enclosed in double quotes (line 1, col 20)
File: good_data.csv
Status: FAILED
  Valid Records:   2
  Invalid Records: 0
  Errors Found (1):
    • Invalid JSON syntax: Expecting property name enclosed in double quotes (line 1, col 20)
File: bad_data.csv
Status: FAILED
  Valid Records:   4
  Invalid Records: 2
  Errors Found (3):
    • Invalid JSON syntax: Expecting property name enclosed in double quotes (line 1, col 20)
    • Row 3: Expected 3 columns, found 2
    • Row 4: Expected 3 columns, found 4


---
## Exercise 5: Format Converter & Exporter
**Objective:** Convert data between JSON and CSV formats in both directions.

Build a bidirectional converter that can transform data files between JSON and CSV. This is an extremely common real-world task — APIs return JSON, but spreadsheets need CSV, and vice versa.

**Tasks:**
1. `json_to_csv(json_filename, csv_filename)` → Read a JSON file (list of dicts) and write it as a CSV file. Use the keys of the first dict as column headers.
2. `csv_to_json(csv_filename, json_filename)` → Read a CSV file and write it as a JSON file (list of dicts).
3. `export_filtered_csv(json_filename, csv_filename, filter_key, filter_value)` → Read a JSON file, keep only records where `record[filter_key] == filter_value`, and export the filtered data to CSV.
4. Handle edge cases: empty files, missing files, and mismatched structures.

In [17]:
import json
import csv

# --- Setup: create a test JSON file ---
students = [
    {"name": "Alice", "age": 22, "city": "Madrid", "grade": "A"},
    {"name": "Bob", "age": 25, "city": "London", "grade": "B"},
    {"name": "Charlie", "age": 30, "city": "Paris", "grade": "A"},
    {"name": "Diana", "age": 28, "city": "Madrid", "grade": "C"},
    {"name": "Eve", "age": 23, "city": "London", "grade": "A"},
]

with open("students.json", "w") as f:
    json.dump(students, f, indent=2)

print("✅ students.json created.")

✅ students.json created.


In [19]:
import json
import csv

def json_to_csv(json_filename, csv_filename):
    """Convert a JSON file (list of dicts) to a CSV file."""
    # TODO: Load JSON data
    # TODO: Extract fieldnames from the first dict's keys
    # TODO: Write CSV with DictWriter
    # TODO: Handle errors (FileNotFoundError, empty data)
    try:
        with open(json_filename, "r", encoding="utf-8") as file:
            data = json.load(file)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"Error reading JSON file '{json_filename}': {e}")
        return

    if not data or not isinstance(data, list):
        print(f"No valid data records found in '{json_filename}'.")
        return
    
    fieldnames = list(data[0].keys())
    try:
        with open(csv_filename, 'w', newline="") as file:
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(data)
    except:
        print("Error Writing into the CSV")

def csv_to_json(csv_filename, json_filename):
    """Convert a CSV file to a JSON file (list of dicts)."""
    # TODO: Read CSV with DictReader
    # TODO: Convert to list of dicts
    # TODO: Write JSON
    # TODO: Handle errors
    records = []
    try:
        with open(csv_filename, 'r') as file:
            reader = csv.DictReader(file)
            records = list(reader)
    except FileNotFoundError:
        print("CSV File not Found!")
        return

    try:
        with open(json_filename, 'w') as file:
            json.dump(records, file, indent=2)
    except Exception as e:
        print(f"Error {e}")
        return


def export_filtered_csv(json_filename, csv_filename, filter_key, filter_value):
    """Export only records where record[filter_key] == filter_value to CSV."""
    # TODO: Load JSON
    # TODO: Filter records
    # TODO: Write filtered data to CSV
    # TODO: Handle empty results and errors
    try:
        with open(json_filename, "r", encoding="utf-8") as file:
            data = json.load(file)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"Error reading JSON file: {e}")
        return

    if not isinstance(data, list) or not data:
        print("No valid data found in JSON file.")
        return

    filtered_data = [d for d in data if d.get(filter_key) == filter_value]

    if not filtered_data:
        print(f"No records found where '{filter_key}' == '{filter_value}'.")
        return

    try:
        with open(csv_filename, "w", encoding="utf-8", newline="") as file:
            writer = csv.DictWriter(file, fieldnames=list(filtered_data[0].keys()))
            writer.writeheader()
            writer.writerows(filtered_data)
        print(f"Exported {len(filtered_data)} records to '{csv_filename}'.")
    except OSError as e:
        print(f"Error writing to CSV: {e}")

# --- TEST ---
# JSON → CSV
json_to_csv("students.json", "students_export.csv")

# CSV → JSON (round-trip test)
csv_to_json("students_export.csv", "students_roundtrip.json")

# Filtered export: only students from Madrid
export_filtered_csv("students.json", "madrid_students.csv", "city", "Madrid")

# Verify round-trip
with open("students_roundtrip.json", "r") as f:
    roundtrip = json.load(f)
print(f"\nRound-trip data ({len(roundtrip)} records):")
for r in roundtrip:
    print(f"  {r}")

Exported 2 records to 'madrid_students.csv'.

Round-trip data (5 records):
  {'name': 'Alice', 'age': '22', 'city': 'Madrid', 'grade': 'A'}
  {'name': 'Bob', 'age': '25', 'city': 'London', 'grade': 'B'}
  {'name': 'Charlie', 'age': '30', 'city': 'Paris', 'grade': 'A'}
  {'name': 'Diana', 'age': '28', 'city': 'Madrid', 'grade': 'C'}
  {'name': 'Eve', 'age': '23', 'city': 'London', 'grade': 'A'}


---
## Exercise 6: Student Records Database (Capstone)
**Objective:** Build a complete file-based CRUD application combining JSON persistence, CSV import/export, and data validation.

Create a student records management system that stores data in a JSON file and supports importing/exporting CSV. This exercise combines **every concept** from this session.

**Required features:**
1. **[A]dd Student** — Prompt for name, age, email, and grade. Validate all inputs (age must be a positive integer, email must contain `@`, grade must be A/B/C/D/F). Store in JSON.
2. **[L]ist Students** — Display all students in a formatted table.
3. **[S]earch** — Find a student by name (case-insensitive partial match).
4. **[U]pdate** — Modify a student's grade or email by name.
5. **[D]elete** — Remove a student by name.
6. **[I]mport CSV** — Load students from a CSV file, validating each row before adding. Skip invalid rows and report what was skipped.
7. **[E]xport CSV** — Export all current students to a CSV file.
8. **[Q]uit** — Exit the program.

**Data structure (JSON):**
```python
[
    {"name": "Alice", "age": 22, "email": "alice@uni.edu", "grade": "A"},
    {"name": "Bob", "age": 25, "email": "bob@uni.edu", "grade": "B"}
]
```

In [2]:
import json
import csv

DB_FILE = "students_db.json"

# --- PERSISTENCE ---

def load_db(filename):
    """Load student records from JSON. Return [] if missing/corrupted."""
    # TODO: try/except for FileNotFoundError and json.JSONDecodeError
    try:
        with open(filename, 'r') as file:
            data = json.load(file)
            return data
    except (FileNotFoundError, json.JSONDecodeError):
        print("File not found")
        return []

def save_db(filename, students):
    """Save student records to JSON."""
    # TODO: json.dump with indent=2
    try:
        with open(filename, 'w') as file:
            json.dump(students, file, indent=2)
    except FileNotFoundError:
        print("File not found")

# --- VALIDATION ---

def validate_student(name, age, email, grade):
    """Validate student fields. Return (is_valid, error_message)."""
    # TODO: name must be non-empty
    # TODO: age must be a positive integer
    # TODO: email must contain '@'
    # TODO: grade must be one of A, B, C, D, F
    if not str(name).strip():
        return (False, 'Name must be non-empty')
    
    try:
        clean_age = int(age)
        if clean_age <= 0:
            return (False, 'Age must be a positive integer')
    except (ValueError, TypeError):
        return (False, 'Age must be a valid integer')
    
    if '@' not in email:
        return (False, 'Email must contain @')
    valid_grades = {'A', 'B', 'C', 'D', 'F'}
    if grade.strip().upper() not in valid_grades:
        return (False, 'Grade must be one of A, B, C, D, F')

    return (True, '')


# --- CRUD OPERATIONS ---

def add_student(students):
    """Prompt for student info, validate, and add to list."""
    # TODO: Get input, call validate_student, append if valid
    while True:
        name = input("Enter Name:")
        age = input("Enter Age:")
        email = input("Enter Email:")
        grade = input("Enter Grade:")

        isvalid, error = validate_student(name, age, email, grade)
        if isvalid:
            students.append({
                'name': name.strip(),
                'age': int(age),
                'email': email.strip(),
                'grade': grade.strip().upper()
            })
            break
        print(error)

def list_students(students):
    """Display all students in a formatted table."""
    # TODO: Handle empty list
    # TODO: Print formatted table with headers
    if not students:
        print("There are no students in the list!")
        return

    print(f"{'#':<4} {'Name':<15} {'Age':<6} {'Grade':<8} {'Email'}")
    print("-" * 55)
    for idx, s in enumerate(students, start=1):
        print(f"{idx:<4} {s['name']:<15} {s['age']:<6} {s['grade']:<8} {s['email']}")

def search_student(students):
    """Search by name (case-insensitive partial match)."""
    # TODO: Get search term
    # TODO: Filter students whose name contains the term
    if not students:
        print("There are no students in the list!")
        return
    keyword = ""
    while not keyword:
        keyword = input("Enter Keyword:").strip().lower()
        if not keyword:
            print("Search keyword cannot be empty.")
            return

    matches = [s for s in students if keyword in s['name'].lower()]
    if not matches:
        print(f"No Students found matching '{keyword}.")
        return

    print(f"\nFound {len(matches)} matching student(s):")
    print(f"{'#':<4} {'Name':<15} {'Age':<6} {'Grade':<8} {'Email'}")
    print("-" * 55)
    for idx, s in enumerate(matches, start=1):
        print(f"{idx:<4} {s['name']:<15} {s['age']:<6} {s['grade']:<8} {s['email']}")

def update_student(students):
    """Update a student's grade or email by name."""
    # TODO: Find student by name
    # TODO: Ask which field to update
    # TODO: Validate new value before applying
    if not students:
        print("There are no students in the file!")
        return

    name = input("Enter the name of the student to update: ").strip().lower()
    student = next((s for s in students if s['name'].lower() == name), None)
    if not student:
        print(f"No Student found with the name {name}")
        return

    print(f"\nUpdating: {student['name']} | Current Grade: {student['grade']} | Current Email: {student['email']}")
    field = input("Which field do you want to update? ([G]rade / [E]mail): ").strip().upper()

    if field == "G":
        new_grade = input("Enter new grade (A, B, C, D, F): ").strip().upper()
        is_valid, error = validate_student(student["name"], student["age"], student["email"], new_grade)
        if is_valid:
            student["grade"] = new_grade
            print(f"Grade updated to {new_grade} for {student['name']}.")
        else:
            print(f"Update failed: {error}")

    elif field == "E":
        new_email = input("Enter new email: ").strip()
        is_valid, error = validate_student(student["name"], student["age"], new_email, student["grade"])
        if is_valid:
            student["email"] = new_email
            print(f"Email updated to {new_email} for {student['name']}.")
        else:
            print(f"Update failed: {error}")

    else:
        print("Invalid choice. Operation cancelled.")

def delete_student(students):
    """Remove a student by name."""
    # TODO: Find and remove student, confirm before deleting
    if not students:
        print("There are no Students in the file!")
        return
    name = input("Enter the name of the student to delete: ").strip().lower()
    student = next((s for s in students if s['name'].lower() == name), None)
    if not student:
        print(f"There is no student found with the name {name}")
        return
    
    confirm = input(f"Are your sure you want to permanently delete {student['name']} y/N: ").strip().lower()
    if confirm == 'y':
        students.remove(student)
        print(f"Student {name} was successfully deleted")
    else:
        print("Delete Cancelled")

# --- IMPORT / EXPORT ---

def import_csv(students, csv_filename):
    """Import students from CSV, validating each row."""
    # TODO: Read CSV with DictReader
    # TODO: Validate each row with validate_student
    # TODO: Skip invalid rows, print what was skipped
    # TODO: Append valid students to the list
    try:
        with open(csv_filename, "r", encoding="utf-8") as file:
            reader = csv.DictReader(file)
            imported_count = 0
            skipped_count = 0

            for row_idx, row in enumerate(reader, start=2):
                name = row.get("name", "")
                age = row.get("age", "")
                email = row.get("email", "")
                grade = row.get("grade", "")

                is_valid, error = validate_student(name, age, email, grade)
                if is_valid:
                    students.append({
                        "name": name.strip(),
                        "age": int(age),
                        "email": email.strip(),
                        "grade": grade.strip().upper(),
                    })
                    imported_count += 1
                else:
                    skipped_count += 1
                    print(f"  • Skipped row {row_idx} ({name or 'Unknown'}): {error}")

            print(f"\nImport finished: {imported_count} imported, {skipped_count} skipped.")

    except FileNotFoundError:
        print(f"Error: CSV file '{csv_filename}' not found.")
    except Exception as e:
        print(f"Error reading CSV file: {e}")

def export_csv(students, csv_filename):
    """Export all students to a CSV file."""
    # TODO: Write CSV with DictWriter
    # TODO: Handle empty data
    if not students:
        print("No student records available to export.")
        return

    fieldnames = ["name", "age", "email", "grade"]

    try:
        with open(csv_filename, "w", encoding="utf-8", newline="") as file:
            writer = csv.DictWriter(file, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(students)
        print(f"Successfully exported {len(students)} record(s) to '{csv_filename}'.")
    except OSError as e:
        print(f"Error exporting to CSV: {e}")

# --- MAIN MENU ---

def run_student_db():
    """Main menu loop."""
    students = load_db(DB_FILE)
    print(f"Loaded {len(students)} student(s) from database.")

    while True:
        print("\n===== STUDENT RECORDS DATABASE =====")
        print("[A]dd  [L]ist  [S]earch  [U]pdate  [D]elete")
        print("[I]mport CSV  [E]xport CSV  [Q]uit")
        choice = input("> ").strip().lower()

        if choice == "a":
            add_student(students)
            save_db(DB_FILE, students)
        elif choice == "l":
            list_students(students)
        elif choice == "s":
            search_student(students)
        elif choice == "u":
            update_student(students)
            save_db(DB_FILE, students)
        elif choice == "d":
            delete_student(students)
            save_db(DB_FILE, students)
        elif choice == "i":
            csv_file = input("CSV filename to import: ").strip()
            import_csv(students, csv_file)
            save_db(DB_FILE, students)
        elif choice == "e":
            csv_file = input("CSV filename to export to: ").strip()
            export_csv(students, csv_file)
        elif choice == "q":
            save_db(DB_FILE, students)
            print("Database saved. Goodbye!")
            break
        else:
            print("Invalid option. Please try again.")

run_student_db()  # Uncomment to test

File not found
Loaded 0 student(s) from database.

===== STUDENT RECORDS DATABASE =====
[A]dd  [L]ist  [S]earch  [U]pdate  [D]elete
[I]mport CSV  [E]xport CSV  [Q]uit
There are no students in the list!

===== STUDENT RECORDS DATABASE =====
[A]dd  [L]ist  [S]earch  [U]pdate  [D]elete
[I]mport CSV  [E]xport CSV  [Q]uit
Email must contain @
Grade must be one of A, B, C, D, F

===== STUDENT RECORDS DATABASE =====
[A]dd  [L]ist  [S]earch  [U]pdate  [D]elete
[I]mport CSV  [E]xport CSV  [Q]uit
#    Name            Age    Grade    Email
-------------------------------------------------------
1    yojan           23     A        jb@gmail.com

===== STUDENT RECORDS DATABASE =====
[A]dd  [L]ist  [S]earch  [U]pdate  [D]elete
[I]mport CSV  [E]xport CSV  [Q]uit
Invalid option. Please try again.

===== STUDENT RECORDS DATABASE =====
[A]dd  [L]ist  [S]earch  [U]pdate  [D]elete
[I]mport CSV  [E]xport CSV  [Q]uit
Database saved. Goodbye!


In [3]:
# --- Generate a sample CSV for testing the Import feature ---

sample_import = """name,age,email,grade
Frank,21,frank@uni.edu,B
Grace,23,grace@uni.edu,A
Hank,not_a_number,hank@uni.edu,C
Ivy,20,ivy_no_at_sign,A
Jack,22,jack@uni.edu,Z
Karen,24,karen@uni.edu,B"""

with open("import_students.csv", "w", newline="") as f:
    f.write(sample_import)

print("import_students.csv created (3 valid, 3 invalid rows).")

import_students.csv created (3 valid, 3 invalid rows).
